# **Alert: Important Notice**

**Note:** The code in this notebook has been tested and optimized for GPU acceleration. Running this code on a CPU or without CUDA may result in errors and suboptimal performance.

It is highly recommended to use a GPU environment with CUDA support for the best experience. If you encounter any issues, please ensure that your environment meets the necessary hardware requirements.

**Disclaimer:** The code may not function as expected on CPU-only setups and could lead to unexpected errors.

# Initialization
Filter out warnings to set the output to minimum

In [ ]:
import warnings

warnings.filterwarnings('ignore')

Install required packages

In [ ]:
print('Installing the requirements...', end=' ')

# replace 'requirements.txt' with path to the requirements file
!pip install -q -r requirements.txt

print('[Done]')

Define color variables to ensure consistent color scheme across all plots

In [ ]:
colors = ['#a1be95', '#f98866']

# Dataset Read
Read the dataset into a pandas dataframe

In [ ]:
import json
import os
import pandas as pd
import random

In [ ]:
def read_dataset(images_dir, annot_file=None, dataset_fraction=1.0):
    print(f"Reading data from {images_dir.split('/')[-1]}...", end=' ')
    image_paths = os.listdir(images_dir)
    if dataset_fraction != 1.0:
        image_paths = random.sample(image_paths, int(len(image_paths) * dataset_fraction))

    if annot_file:
        with open(annot_file) as f:
            annot_data = json.load(f)

        image_id_to_captions = {}
        for annot in annot_data['annotations']:
            img_id = annot['image_id']
            caption = annot['caption']
            if img_id not in image_id_to_captions:
                image_id_to_captions[img_id] = []
            image_id_to_captions[img_id].append(caption)

        records = []
        for img in annot_data['images']:
            img_filename = img['file_name']
            img_id = img['id']
            if img_filename in image_paths:
                captions = image_id_to_captions.get(img_id, [])
                records.append([img_filename, captions])

        df = pd.DataFrame(records, columns=['image_path', 'captions'])
    else:
        # For test dataset, only return image paths
        df = pd.DataFrame(image_paths, columns=['image_path'])

    print('[Done]')
    return df

In [ ]:
df_train = read_dataset(
    images_dir='COCO/train_subset',
    annot_file='COCO/annotations/captions_train2017.json',
    dataset_fraction=1.0
)
df_val = read_dataset(
    images_dir='COCO/val_subset',
    annot_file='COCO/annotations/captions_val2017.json',
    dataset_fraction=1.0
)

Preview the datasets

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, UnidentifiedImageError

In [ ]:
def show_image_previews(df, directory, num_images=5, target_size=(224, 224)):
    if 'captions' in df.columns:
        # Set up a figure with subplots, one row per image, 2 columns (image and captions)
        fig, axes = plt.subplots(num_images, 2, figsize=(9, num_images * 1.5))

        # Randomly sample num_images rows from the DataFrame
        sample_df = df[['image_path', 'captions']].sample(n=num_images)

        for idx, (image_path, captions) in enumerate(sample_df.values):
            try:
                # Open the image
                image = Image.open(directory + image_path)
                image = ImageOps.fit(image, target_size)

                # Display the image
                axes[idx, 0].imshow(image)
                axes[idx, 0].axis('off')
            except UnidentifiedImageError:
                pass

            for index in range(len(captions)):
                if len(captions[index]) > 60:
                    captions[index] = captions[index][:60] + '...'

            # Display the captions
            axes[idx, 1].text(0, 0.5, '\n'.join(captions), fontsize=12, va='center', ha='left')
            axes[idx, 1].axis('off')

        plt.suptitle(f"Preview of {directory.split('/')[1]}", fontsize=16, y=1.02)
    else:
        sample_df = df['image_path'].sample(n=num_images)

        fig, axes = plt.subplots(1, num_images, figsize=(15, 5))

        for ax, img_path in zip(axes, sample_df):
            try:
                img = Image.open(directory + img_path)
                img = ImageOps.fit(img, target_size)
                ax.imshow(img)
                ax.axis('off')
            except UnidentifiedImageError:
                pass

        plt.suptitle(f"Preview of {directory.split('/')[1]}", fontsize=22, y=0.9)

    plt.tight_layout()
    plt.show()

In [ ]:
print(f'Size of train dataset: {len(df_train):,}')
df_train.head()

In [ ]:
show_image_previews(df_train, 'COCO/train_subset/')

In [ ]:
print(f'Size of val dataset: {len(df_val):,}')
df_val.head()

In [ ]:
show_image_previews(df_val, 'COCO/val_subset/')

Expand each row to multiple rows with one image and one caption

In [ ]:
def expand_dataframe(df):
    expanded_rows = []
    for _, row in df.iterrows():
        image_path = row['image_path']
        captions = row['captions']
        for caption in captions:
            expanded_rows.append({'image_path': image_path, 'caption': caption})
    return pd.DataFrame(expanded_rows).sample(frac=1).reset_index(drop=True)

In [ ]:
df_train = expand_dataframe(df_train)
df_val = expand_dataframe(df_val)

Preview the expanded data

In [ ]:
print(f'Size of train dataset: {len(df_train):,}')
df_train.head(10)

In [ ]:
print(f'Size of val dataset: {len(df_val):,}')
df_val.head(10)

In [ ]:
sizes = [len(df_train), len(df_val)]
labels = ['Train', 'Validation']

plt.figure(figsize=(7, 7))

# Create pie chart
wedges, texts, auto_texts = plt.pie(
    sizes, labels=labels, autopct='%1.1f%%', startangle=90,
    colors=[colors[0], colors[1]]
)
for autotext in auto_texts:
    autotext.set_color('white')
for wedge in wedges:
    wedge.set_edgecolor('white')

plt.title('Percentage of Train and Validation Datasets', y=1.05)
plt.axis('equal')
plt.show()

# Build Vocabulary

In [ ]:
import nltk
from collections import Counter

nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)

stop_words = set(nltk.corpus.stopwords.words('english'))

In [ ]:
def tokenize_caption(caption):
    tokens = nltk.tokenize.word_tokenize(caption)
    tokens = [token.lower() for token in tokens if token.isalpha() and token.lower() not in stop_words]
    return tokens

In [ ]:
def build_vocab(captions, min_freq=5):
    word_count = Counter(token for caption in captions for token in caption)
    idx2token = {0: '<PAD>', 1: '<START>', 2: '<END>', 3: '<UNK>'}
    token2idx = {v: k for k, v in idx2token.items()}

    idx = 4
    for word, count in word_count.items():
        if count >= min_freq:
            idx2token[idx] = word
            token2idx[word] = idx
            idx += 1

    return {'idx2token': idx2token, 'token2idx': token2idx, 'word_count': word_count}

Tokenize all captions

In [ ]:
print('Tokenizing captions...', end=' ')
df_train['caption_tokenized'] = df_train['caption'].apply(tokenize_caption)
df_val['caption_tokenized'] = df_val['caption'].apply(tokenize_caption)
print('[Done]')

In [ ]:
df_train.head()

In [ ]:
df_val.head()

Build vocabulary from train captions

In [ ]:
print('Building vocabulary...', end=' ')
vocab = build_vocab(captions=df_train['caption_tokenized'])
print('[Done]')
print(f"Size of vocabulary: {len(vocab['idx2token']):,}")

Show preview of vocabulary

In [ ]:
from wordcloud import WordCloud

In [ ]:
# noinspection PyUnusedLocal
def color_func(word, font_size, position, orientation, random_state=None, **kwargs):
    return random.choice(colors)

In [ ]:
wordcloud = WordCloud(
    width=800, height=400, background_color='white', color_func=color_func
).generate_from_frequencies(vocab['word_count'])

In [ ]:
plt.figure(figsize=(10, 5))
plt.imshow(wordcloud, interpolation='bilinear')
plt.axis('off')
plt.show()

# Create Graph Data
Create pytorch geometric data and data loaders

In [ ]:
import torch
from torch_geometric.data import Data

Create graph data for each caption

In [ ]:
def get_caption_graph(caption, vocab, image_path=None):
    encoded_caption = [vocab['token2idx'].get(token, vocab['token2idx']['<UNK>']) for token in caption]
    encoded_caption.insert(0, vocab['token2idx']['<START>'])
    encoded_caption.append(vocab['token2idx']['<END>'])

    caption_seq = list(range(len(encoded_caption)))

    # Neighborhood creation (graph edges) using sliding window
    window_size = 3
    num_tokens = len(caption_seq)
    neighbourhood = {i: set() for i in caption_seq}

    for i in range(num_tokens - window_size + 1):
        window = caption_seq[i:i + window_size]
        for word in window:
            neighbourhood[word].update(window)

    edge_index_1, edge_index_2 = [], []
    for src, neighbours in neighbourhood.items():
        for dest in neighbours:
            if src != dest:
                edge_index_1.append(src)
                edge_index_2.append(dest)

    caption_seq_tensor = torch.tensor(caption_seq, dtype=torch.long)
    encoded_caption_tensor = torch.tensor(encoded_caption, dtype=torch.long)
    edge_index = torch.tensor([edge_index_1, edge_index_2], dtype=torch.long)

    if image_path is None:
        return Data(x=caption_seq_tensor, edge_index=edge_index, caption=encoded_caption_tensor)
    else:
        return Data(x=caption_seq_tensor, edge_index=edge_index, image_path=image_path, caption=encoded_caption_tensor)

Convert raw data to graphs

In [ ]:
def create_graph_data(image_paths, captions=None, vocab=None):
    # If captions are provided
    if captions is not None:
        graphs = [get_caption_graph(caption=caption, vocab=vocab, image_path=image_path)
                  for image_path, caption in zip(image_paths, captions)]
    # If only image paths are provided 
    else:
        graphs = [Data(image_path=image_path) for image_path in image_paths]

    return graphs

In [ ]:
print('Creating graph data...', end=' ')
train_graphs = create_graph_data(
    image_paths=df_train['image_path'], captions=df_train['caption_tokenized'], vocab=vocab
)
val_graphs = create_graph_data(
    image_paths=df_val['image_path'], captions=df_val['caption_tokenized'], vocab=vocab
)
print('[Done]')

Visualize a sample of graphs

In [ ]:
import networkx as nx
from torch_geometric.utils import to_networkx

In [ ]:
def visualize_graph(data):
    G = to_networkx(data)
    pos = nx.spring_layout(G)
    nx.draw(G, pos, with_labels=False, node_color=colors[0])
    nx.draw_networkx_labels(G, pos, font_color='white')

    # Decode the caption
    caption_indices = data.caption.tolist()
    caption_tokens = [vocab['idx2token'][idx] for idx in caption_indices]
    caption_with_indices = ' | '.join([f"{idx}. {token}" for token, idx in zip(caption_tokens, data.x.tolist())])

    plt.suptitle(caption_with_indices, y=1.05)
    plt.show()

In [ ]:
visualize_graph(random.sample(train_graphs, 1)[0])

# Image Encoder
Image encoder architecture with use of pretrained ResNet50

In [ ]:
import torch.nn as nn
from torchvision.models import resnet50

In [ ]:
class ImageEncoder(nn.Module):
    def __init__(self, config):
        super(ImageEncoder, self).__init__()
        # Extract ResNet50 backbone without the final classification layer
        resnet = resnet50(weights='ResNet50_Weights.DEFAULT', progress=False)
        self.model = nn.Sequential(*list(resnet.children())[:-1])

        for param in self.model.parameters():
            param.requires_grad = False

        self.fc = nn.Sequential(
            nn.Linear(in_features=2048, out_features=config.embed_dim),
            nn.ReLU(),
            nn.BatchNorm1d(num_features=config.embed_dim),
            nn.Dropout(p=config.dropout_p),
            nn.Linear(in_features=config.embed_dim, out_features=config.embed_dim),
            nn.BatchNorm1d(num_features=config.embed_dim)
        )

    def forward(self, images):
        # images: [batch_size, 3, 224, 224]
        features = self.model(images)
        # features: [batch_size, 2048, 1, 1]
        if features.shape[0] != 1:
            features = features.squeeze()
        else:
            features = features.squeeze(-1).squeeze(-1)
        # features: [batch_size, 2048]
        output = self.fc(features)
        # output: [batch_size, embed_dim]
        return output

# Text Encoder
Proposed text encoder architecture using GNNs

In [ ]:
import torch_geometric.nn as gnn
from torch.nn.utils.rnn import pad_sequence, pack_padded_sequence, pad_packed_sequence

In [ ]:
class TextEncoder(nn.Module):
    def __init__(self, config):
        super(TextEncoder, self).__init__()
        self.config = config
        self.embedding = nn.Embedding(num_embeddings=config.vocab_len, embedding_dim=config.embed_dim)

        self.rnn = nn.LSTM(
            input_size=config.embed_dim, hidden_size=config.embed_dim,
            bidirectional=True, batch_first=True
        )

        self.gcn1 = gnn.SAGEConv(in_channels=config.embed_dim * 2, out_channels=config.embed_dim)
        self.gcn2 = gnn.SAGEConv(in_channels=config.embed_dim, out_channels=config.embed_dim)
        self.gcn3 = gnn.SAGEConv(in_channels=config.embed_dim, out_channels=config.embed_dim)

        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p=config.dropout_p)

    # Graph data preparation
    def prepare(self, graph_data):
        batch2nodes = {}
        for idx in range(self.config.batch_size):
            batch2nodes[idx] = []

        for node, batch in zip(graph_data.caption, graph_data.batch):
            batch2nodes[batch.item()].append(node.item())

        seq_list = [torch.tensor(batch2nodes[item], dtype=torch.long) for item in batch2nodes]
        seq_lens = [len(batch2nodes[item]) for item in batch2nodes]

        # Using pad_sequence to avoid manual padding handling
        seq_list_padded = pad_sequence(
            sequences=seq_list, batch_first=True, padding_value=self.config.padding_value
        )
        return seq_list_padded, seq_lens

    def forward(self, graphs):
        # graphs: [batch_size]
        self.config.batch_size = len(graphs)
        seq_list_padded, seq_lens = self.prepare(graph_data=graphs)
        seq_list_padded = seq_list_padded.to(self.config.device, non_blocking=True)
        # seq_list_padded: [batch_size, padded_seq_len]
        seq_features = self.embedding(seq_list_padded)
        # seq_features: [batch_size, padded_seq_len, embed_dim]
        packed_input = pack_padded_sequence(seq_features, seq_lens, batch_first=True, enforce_sorted=False)
        rnn_features, _ = self.rnn(packed_input)
        rnn_features, _ = pad_packed_sequence(rnn_features, batch_first=True)
        # rnn_features: [batch_size, padded_seq_len, embed_dim * 2]
        rnn_features = [rnn_features[idx, 0:item, :] for idx, item in enumerate(seq_lens)]
        rnn_features = torch.cat(rnn_features, dim=0)
        # rnn_features: [sum(seq_lens), embed_dim * 2]
        features = self.gcn1(rnn_features, graphs.edge_index)
        features = self.relu(features)
        features = self.gcn2(features, graphs.edge_index)
        features = self.dropout(features)
        features = self.gcn3(features, graphs.edge_index)
        # features: [sum(seq_lens), embed_dim]
        output = gnn.global_mean_pool(x=features, batch=graphs.batch)
        # output: [batch_size, embed_dim]
        return output

# GNN-IR Model
Main model including image encoder and text encoder

In [ ]:
from concurrent.futures import ThreadPoolExecutor
from torchvision import transforms
from torchvision.models import ResNet50_Weights

In [ ]:
class GNN_IR(nn.Module):
    def __init__(self, config):
        super(GNN_IR, self).__init__()
        self.config = config
        self.transform = transforms.Compose([
            transforms.PILToTensor(),
            ResNet50_Weights.IMAGENET1K_V2.transforms(antialias=True)
        ])
        self.image_encoder = ImageEncoder(config=config)
        self.text_encoder = TextEncoder(config=config)

    # Read batch images from dataset
    def read_images(self, image_paths):
        def load_and_transform(image_path):
            dir_train = os.path.join('COCO/train_subset', image_path)
            dir_val = os.path.join('COCO/val_subset', image_path)

            try:
                if os.path.isfile(dir_train):
                    image_pil = Image.open(dir_train)
                elif os.path.isfile(dir_val):
                    image_pil = Image.open(dir_val)
                else:
                    raise FileNotFoundError(f'No image found: {image_path}')

                # Convert and transform the image
                image_pil = image_pil.convert('RGB')
                return self.transform(image_pil)
            except (UnidentifiedImageError, FileNotFoundError):
                # Return a placeholder tensor if image loading fails
                return torch.zeros(3, 224, 224)

        # Using ThreadPoolExecutor for parallel image loading
        with ThreadPoolExecutor() as executor:
            images = list(executor.map(load_and_transform, image_paths))

        # Filter out None if there are any cases where loading failed entirely
        valid_images = [img for img in images if img is not None]

        # Ensure the batch is not empty
        if len(valid_images) == 0:
            raise ValueError("All images failed to load!")

        return torch.stack(valid_images)

    def forward(self, graphs):
        # graphs: [batch_size]
        images = self.read_images(graphs.image_path).to(self.config.device, non_blocking=True)
        # images: [batch_size, 3, 224, 224]
        image_embeddings = self.image_encoder(images)
        # image_embeddings: [batch_size, embed_dim]
        caption_embeddings = self.text_encoder(graphs)
        # caption_embeddings: [batch_size, embed_dim]
        return image_embeddings, caption_embeddings

# Loss Function
Symmetric loss function derived from CLIP paper

In [ ]:
import torch.nn.functional as F

In [ ]:
def symmetric_loss(logit, batch_size, device):
    labels = torch.arange(batch_size).to(device, non_blocking=True)

    loss_image = F.cross_entropy(logit, labels, reduction='mean')
    loss_text = F.cross_entropy(logit.T, labels, reduction='mean')

    loss = (loss_image + loss_text) / 2
    return loss

# Train and Validation
Function to train and validate the model for one epoch

In [ ]:
from torch import autocast
from torch.nn.utils import clip_grad_norm_
from tqdm import tqdm

In [ ]:
def train_one_epoch(model, train_loader, optimizer, scaler, config):
    model.train()

    batch_loss = 0.0
    for graphs in tqdm(train_loader):
        optimizer.zero_grad()

        graphs = graphs.to(config.device, non_blocking=True)

        with autocast(device_type=config.device.type, dtype=torch.float16):
            image_embeddings, caption_embeddings = model(graphs)
            # image_embeddings : [batch_size, embed_dim]
            # caption_embeddings : [batch_size, embed_dim]

            logit = torch.matmul(image_embeddings, caption_embeddings.T)
            loss = symmetric_loss(logit, batch_size=len(graphs.image_path), device=config.device)

            batch_loss += loss.item()

        # loss.backward()
        scaler.scale(loss).backward()

        # gradient clip
        scaler.unscale_(optimizer)
        clip_grad_norm_(parameters=model.parameters(), max_norm=config.gradient_clip)

        # optimizer.step()
        scaler.step(optimizer)
        scaler.update()

    avg_loss = batch_loss / len(train_loader)

    return avg_loss

In [ ]:
def val_one_epoch(model, val_loader, config):
    model.eval()

    batch_loss = 0.0
    for graphs in tqdm(val_loader):
        graphs = graphs.to(config.device, non_blocking=True)

        with torch.no_grad():
            with autocast(device_type=config.device.type, dtype=torch.float16):
                image_embeddings, caption_embeddings = model(graphs)
                # image_embeddings : [batch_size, embed_dim]
                # caption_embeddings : [batch_size, embed_dim]

                logit = torch.matmul(image_embeddings, caption_embeddings.T)
                loss = symmetric_loss(logit, batch_size=len(graphs.image_path), device=config.device)

                batch_loss += loss.item()

    avg_loss = batch_loss / len(val_loader)

    return avg_loss

## Configuration
Create configuration class for parameters and hyperparameters

In [ ]:
class Config:
    def __init__(self):
        self.batch_size = 32
        self.cuda_device_name = torch.cuda.get_device_name() if torch.cuda.is_available() else 'CPU'
        self.device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
        self.dropout_p = 0.45
        self.embed_dim = 1024
        self.gamma = 0.65
        self.gradient_clip = 1.0
        self.learning_rate = 1.9e-5
        self.padding_value = 0
        self.scheduler_step_size = 6
        self.vocab_len = len(vocab['idx2token'])

    def to_dict(self):
        return {
            'batch_size': self.batch_size,
            'cuda_device_name': self.cuda_device_name,
            'device': str(self.device),
            'dropout_p': self.dropout_p,
            'embed_dim': self.embed_dim,
            'gamma': self.gamma,
            'gradient_clip': self.gradient_clip,
            'learning_rate': self.learning_rate,
            'padding_value': self.padding_value,
            'scheduler_step_size': self.scheduler_step_size,
            'vocab_len': self.vocab_len
        }

In [ ]:
config = Config()

Initialize model and its training

In [ ]:
from torch.cuda.amp import GradScaler
from torch.optim import AdamW
from torch.optim.lr_scheduler import StepLR
from torch_geometric.loader import DataLoader

In [ ]:
train_loader = DataLoader(train_graphs, batch_size=config.batch_size, shuffle=True)
val_loader = DataLoader(val_graphs, batch_size=config.batch_size, shuffle=False)

In [ ]:
model = GNN_IR(config=config).to(config.device, non_blocking=True)
optimizer = AdamW(params=model.parameters(), lr=config.learning_rate)
scheduler = StepLR(optimizer, step_size=config.scheduler_step_size, gamma=config.gamma)
scaler = GradScaler()

Train the model

In [ ]:
import datetime
import time

In [ ]:
model_states = []
train_losses = []
val_losses = []

best_loss = float('inf')
early_stop_patience = 4
epochs = 100
epoch_limit = epochs
epoch_times = []
time_limit = 36000  # 10 hours
violations = 0

print(f'Model training on {config.cuda_device_name}')
for epoch in range(epochs):
    print(f'Epoch: {epoch}')
    start_time = time.time()

    train_loss = train_one_epoch(
        model=model,
        train_loader=train_loader,
        optimizer=optimizer,
        scaler=scaler,
        config=config
    )
    val_loss = val_one_epoch(
        model=model,
        val_loader=val_loader,
        config=config
    )

    end_time = time.time()
    epoch_time = end_time - start_time
    epoch_times.append(epoch_time)

    # Print information of epoch
    print(f"Time spent: {str(datetime.timedelta(seconds=epoch_time)).split('.')[0]}")
    print(f'Train Loss: {train_loss:.3f} | Validation Loss: {val_loss:.3f}')

    if epoch == 2:
        avg_epoch_time = sum(epoch_times) / len(epoch_times)
        print(f"Average Epoch Time: {str(datetime.timedelta(seconds=avg_epoch_time)).split('.')[0]}")
        for i in range(epochs):
            if avg_epoch_time * (i + 1) > time_limit:
                epoch_limit = i
                print(f'Model training and validation will stop on epoch {epoch_limit} due to time limit.')
                break

    if val_loss < best_loss:
        best_loss = val_loss
        violations = 0
    else:
        violations += 1
        if violations > early_stop_patience:
            print("Early stopping triggered.")
            break

    model_states.append(model.state_dict())
    train_losses.append(train_loss)
    val_losses.append(val_loss)
    scheduler.step()

    if epoch == epoch_limit:
        break

Plot loss over epochs

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(range(len(train_losses)), train_losses, label='Training Loss', color=colors[0], marker='o')
plt.plot(range(len(val_losses)), val_losses, label='Validation Loss', color=colors[1], marker='o')

plt.xlabel('Epochs')
plt.ylabel('Loss')
plt.xticks(range(len(train_losses)))
plt.title('Training and Validation Losses Over Epochs')
plt.legend()
plt.show()

Revert to the best model based on loss values

In [ ]:
best_model_index = None
lowest_combined_loss = float('inf')

for i in range(len(model_states)):
    combined_loss = train_losses[i] + val_losses[i]
    if combined_loss < lowest_combined_loss:
        lowest_combined_loss = combined_loss
        best_model_index = i
best_model_state = model_states[best_model_index]
model.load_state_dict(best_model_state)

print(f'Best model state on epoch: {best_model_index}')
print(f'Combined loss on best state of model: {lowest_combined_loss:.3f}')

# Evaluation
Calculate the accuracy and mAP of model using train and val data

In [ ]:
from sklearn.metrics import average_precision_score

In [ ]:
def get_embeddings(model, data_loader, config):
    model.eval()

    image_embeddings = []
    caption_embeddings = []

    with torch.no_grad():
        with autocast(device_type=config.device.type, dtype=torch.float16):
            for graphs in tqdm(data_loader):
                graphs = graphs.to(config.device, non_blocking=True)

                image_embedding, caption_embedding = model(graphs)

                # Move to CPU to save GPU memory
                image_embeddings.append(image_embedding.cpu())
                caption_embeddings.append(caption_embedding.cpu())

    image_embeddings = torch.cat(image_embeddings)
    caption_embeddings = torch.cat(caption_embeddings)

    # Normalize the embeddings
    image_embeddings_normalized = image_embeddings / torch.norm(image_embeddings, dim=1, keepdim=True)
    caption_embeddings_normalized = caption_embeddings / torch.norm(caption_embeddings, dim=1, keepdim=True)

    return image_embeddings_normalized, caption_embeddings_normalized

In [ ]:
def evaluate_model(image_embeddings, caption_embeddings, k, captions, batch_size):
    num_captions = caption_embeddings.shape[0]
    topk_acc_counter = 0
    ap_sum = 0.0

    for start_idx in tqdm(range(0, num_captions, batch_size)):
        end_idx = min(start_idx + batch_size, num_captions)

        similarity_matrix = torch.matmul(caption_embeddings[start_idx:end_idx], image_embeddings.T)

        for i in range(similarity_matrix.shape[0]):
            _, retrieved_indices = torch.topk(similarity_matrix[i], k=k, largest=True)

            # Calculate ground truth relevance
            results = [
                [token in captions[start_idx + i] for token in captions[j]]
                for j in retrieved_indices.tolist()
            ]
            relevant_labels = torch.tensor([result.count(True) >= 2 for result in results], dtype=torch.float32)

            # Top-K accuracy
            if (start_idx + i) in retrieved_indices or relevant_labels.any():
                topk_acc_counter += 1

            # mAP calculation
            if relevant_labels.any():
                ap_sum += average_precision_score(relevant_labels.numpy(), retrieved_indices.numpy())

    topk_acc = topk_acc_counter / num_captions
    topk_map = ap_sum / num_captions

    return topk_acc, topk_map

In [ ]:
# Reset configuration to undo changes from training/validation
config = Config()

In [ ]:
evaluation_results = {
    'train': {'topk_accs': {}, 'topk_maps': {}},
    'validation': {'topk_accs': {}, 'topk_maps': {}},
    'config': config.to_dict()
}

k_values = [1, 5, 10, 15]

Create image/caption embeddings of train data using trained model

In [ ]:
from sklearn.decomposition import PCA

In [ ]:
train_loader = DataLoader(train_graphs, batch_size=config.batch_size, shuffle=False)
image_embeddings, caption_embeddings = get_embeddings(model, data_loader=train_loader, config=config)
image_embeddings_np = image_embeddings.cpu().numpy()
caption_embeddings_np = caption_embeddings.cpu().numpy()

In [ ]:
pca = PCA(n_components=2)
image_embeddings_2d = pca.fit_transform(image_embeddings_np)
caption_embeddings_2d = pca.fit_transform(caption_embeddings_np)

plt.figure(figsize=(8, 8))
plt.scatter(
    image_embeddings_2d[:, 0], image_embeddings_2d[:, 1],
    label='Image Embeddings', alpha=0.6, marker='o', color=colors[0]
)
plt.scatter(
    caption_embeddings_2d[:, 0], caption_embeddings_2d[:, 1],
    label='Caption Embeddings', alpha=0.6, marker='x', color=colors[1]
)

plt.title('2D PCA Projection of Train Image and Caption Embeddings')
plt.legend()
plt.show()

Evaluate the training phase of model

In [ ]:
print('Evaluation of model training:')
for k in k_values:
    topk_acc, topk_map = evaluate_model(
        image_embeddings,
        caption_embeddings,
        k=k,
        captions=df_train['caption_tokenized'],
        batch_size=config.batch_size
    )

    evaluation_results['train']['topk_accs'][k] = topk_acc
    evaluation_results['train']['topk_maps'][k] = topk_map

    print(f'Train accuracy with K={k}: {topk_acc:.3f}')
    print(f'Train mAP with K={k}: {topk_map:.3f}')

Create image/caption embeddings of validation data using trained model

In [ ]:
val_loader = DataLoader(val_graphs, batch_size=config.batch_size, shuffle=False)
image_embeddings, caption_embeddings = get_embeddings(model, data_loader=val_loader, config=config)
image_embeddings_np = image_embeddings.cpu().numpy()
caption_embeddings_np = caption_embeddings.cpu().numpy()

In [ ]:
pca = PCA(n_components=2)
image_embeddings_2d = pca.fit_transform(image_embeddings_np)
caption_embeddings_2d = pca.fit_transform(caption_embeddings_np)

plt.figure(figsize=(8, 8))
plt.scatter(
    image_embeddings_2d[:, 0], image_embeddings_2d[:, 1],
    label='Image Embeddings', alpha=0.6, marker='o', color=colors[0]
)
plt.scatter(
    caption_embeddings_2d[:, 0], caption_embeddings_2d[:, 1],
    label='Caption Embeddings', alpha=0.6, marker='x', color=colors[1]
)

plt.title('2D PCA Projection of Validation Image and Caption Embeddings')
plt.legend()
plt.show()

Evaluate the validation phase of model

In [ ]:
print('Evaluation of model validation:')
for k in k_values:
    topk_acc, topk_map = evaluate_model(
        image_embeddings,
        caption_embeddings,
        k=k,
        captions=df_val['caption_tokenized'],
        batch_size=config.batch_size
    )

    evaluation_results['validation']['topk_accs'][k] = topk_acc
    evaluation_results['validation']['topk_maps'][k] = topk_map

    print(f'Validation accuracy with K={k}: {topk_acc:.3f}')
    print(f'Validation mAP with K={k}: {topk_map:.3f}')

Plot the model evaluation results

In [ ]:
plt.figure(figsize=(10, 5))

plt.subplot(1, 2, 1)
plt.plot(
    k_values,
    [evaluation_results['train']['topk_accs'][k] for k in k_values],
    label='Train Accuracy',
    color=colors[0],
    marker='o'
)
plt.plot(
    k_values,
    [evaluation_results['validation']['topk_accs'][k] for k in k_values],
    label='Validation Accuracy',
    color=colors[1],
    marker='o'
)
plt.title('Top-K Accuracy')
plt.xlabel('K')
plt.ylabel('Accuracy')
plt.xticks(k_values)
plt.legend()

plt.subplot(1, 2, 2)
plt.plot(
    k_values,
    [evaluation_results['train']['topk_maps'][k] for k in k_values],
    label='Train mAP',
    color=colors[0],
    marker='o'
)
plt.plot(
    k_values,
    [evaluation_results['validation']['topk_maps'][k] for k in k_values],
    label='Validation mAP',
    color=colors[1],
    marker='o'
)
plt.title('Top-K mAP')
plt.xlabel('K')
plt.ylabel('Mean Average Precision')
plt.xticks(k_values)
plt.legend()

plt.tight_layout()
plt.show()

# Save Model
Save model state, configuration and training/validation info

In [ ]:
current_time = datetime.datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

In [ ]:
print('Saving model and info...', end=' ')
os.makedirs('versions', exist_ok=True)
torch.save(model.state_dict(), f'versions/model_{current_time}.pt')
with open(f'versions/results_{current_time}.json', 'w') as json_file:
    json.dump(evaluation_results, json_file, indent=4)
print('[Done]')

# Test the Model
Read the test dataset

In [ ]:
df_test = read_dataset(images_dir='COCO/test_subset', dataset_fraction=1.0)

In [ ]:
print(f'Size of test dataset: {len(df_test):,}')
df_test.head()

In [ ]:
show_image_previews(df_test, 'COCO/test_subset/')

Get image embeddings

In [ ]:
def get_image_embeddings(model, data_loader, config):
    model.eval()

    image_embeddings = torch.zeros((len(df_test), config.embed_dim), device=config.device, dtype=torch.float16)
    with torch.no_grad():
        with autocast(device_type=config.device.type, dtype=torch.float16):
            start_idx = 0
            for graphs in tqdm(data_loader):
                batch_size = len(graphs.image_path)
                images = model.read_images(graphs.image_path).to(config.device)
                batch_image_embeddings = model.image_encoder(images)
                image_embeddings[start_idx:start_idx + batch_size] = batch_image_embeddings

                start_idx += batch_size

    image_embeddings_normalized = image_embeddings / torch.norm(image_embeddings, dim=1, keepdim=True)

    return image_embeddings_normalized

Prepare test data

In [ ]:
print('Testing the model:')
test_graphs = create_graph_data(image_paths=df_test['image_path'])
test_loader = DataLoader(test_graphs, batch_size=config.batch_size, shuffle=False)
test_image_embeddings = get_image_embeddings(model, data_loader=test_loader, config=config)

Find matches to given query

In [ ]:
def find_matches(model, df, vocab, query, k):
    tokenized_query = tokenize_caption(query)
    query_graph = get_caption_graph(caption=tokenized_query, vocab=vocab)
    query_loader = DataLoader([query_graph], batch_size=1, shuffle=False)

    model.eval()
    with torch.no_grad():
        with autocast(device_type=config.device.type, dtype=torch.float16):
            for q in query_loader:
                q = q.to(config.device, non_blocking=True)
                query_embedding = model.text_encoder(q)

            # Normalize the embeddings
            query_embedding = query_embedding / torch.norm(query_embedding, dim=1, keepdim=True)

            dot_similarity = torch.matmul(query_embedding, test_image_embeddings.T)
            _, indices = torch.topk(dot_similarity.squeeze(0), k=k)

            matches = df.iloc[indices.tolist()]['image_path'].tolist()

    return matches

In [ ]:
def retrieve_images(query, k=5, directory='COCO/test_subset/', target_size=(224, 224)):
    matches = find_matches(model, df=df_test, vocab=vocab, query=query, k=k)
    _, axes = plt.subplots(1, k, figsize=(10, 10))

    def load_and_display_image(match, ax):
        try:
            # Open the image and resize
            image = Image.open(directory + match)
            image = ImageOps.fit(image, target_size)
            # Display the image
            ax.imshow(image)
            ax.axis('off')
        except UnidentifiedImageError:
            pass

    # Use ThreadPoolExecutor for parallel image loading and display
    with ThreadPoolExecutor() as executor:
        for idx, (match, ax) in enumerate(zip(matches, axes)):
            executor.submit(load_and_display_image, match, ax)

    plt.show()

In [ ]:
retrieve_images(query='Two men seating in a park')

In [ ]:
retrieve_images(query='A girl playing with a cat')